# Data exploration

Inspect images, annotations, class balance, image dimensions, corrupt files, and official splits. Decide classification or segmentation and an evaluation protocol before training.

Setup only. Implementation will be added one learning step at a time.

In [2]:

import sys
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Device:", device)

assert device.type == "cuda", "GPU is not enabled!"


Python: /Users/farabi/Farabi/ML-Projects/CV/Oxford-IIT Pet/.venv/bin/python
PyTorch: 2.14.1
Device: cpu


AssertionError: GPU is not enabled!

In [12]:
from datasets import load_dataset

dataset = load_dataset("timm/oxford-iiit-pet")

In [13]:
from tqdm.auto import tqdm

In [14]:
#Checking the structure of the dataset,what labels are there and what splits it has ,
#It doesnt have validation split, it only has test and train dataset


print(dataset)
print(dataset.keys)
split_name = next(iter(dataset.keys()))
print(dataset[split_name].features)


DatasetDict({
    train: Dataset({
        features: ['image', 'label', 'image_id', 'label_cat_dog'],
        num_rows: 3680
    })
    test: Dataset({
        features: ['image', 'label', 'image_id', 'label_cat_dog'],
        num_rows: 3669
    })
})
<built-in method keys of DatasetDict object at 0x1258c32a0>
{'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['abyssinian', 'american_bulldog', 'american_pit_bull_terrier', 'basset_hound', 'beagle', 'bengal', 'birman', 'bombay', 'boxer', 'british_shorthair', 'chihuahua', 'egyptian_mau', 'english_cocker_spaniel', 'english_setter', 'german_shorthaired', 'great_pyrenees', 'havanese', 'japanese_chin', 'keeshond', 'leonberger', 'maine_coon', 'miniature_pinscher', 'newfoundland', 'persian', 'pomeranian', 'pug', 'ragdoll', 'russian_blue', 'saint_bernard', 'samoyed', 'scottish_terrier', 'shiba_inu', 'siamese', 'sphynx', 'staffordshire_bull_terrier', 'wheaten_terrier', 'yorkshire_terrier']), 'image_id': Value('string'), 'label_ca

In [15]:
sample = dataset[split_name][0]
print(sample.keys())

dict_keys(['image', 'label', 'image_id', 'label_cat_dog'])


In [16]:
'''
We do not have validation ds, so we will create a internal validation ds from the train set
we will use internal test set to fine tune our model's weights and run experiments
'''

split = dataset["train"].train_test_split(
    test_size = 0.2,
    seed = 42,
    stratify_by_column="label"
)


train_ds = split["train"]
val_ds = split["test"]
test_ds = dataset["test"]

print("Training:", len(train_ds))
print("Validation:", len(val_ds))
print("Testing:", len(test_ds))

Training: 2944
Validation: 736
Testing: 3669


In [17]:
from torchvision import transforms

#transform the train_data to do data augmentation which will help the model in future to generalize 


train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        size = 224,
        scale = (0.8,1.0)
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], #from ImageNet statistics
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = val_transform


In [18]:
import torch
from torch.utils.data import Dataset,DataLoader

class PetDataset(Dataset):
    def __init__(self,hf_dataset,transform=None):
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self,idx):
        sample = self.dataset[idx]

        image = sample["image"].convert("RGB")
        label = sample["label"]

        if self.transform is not None:
            image = self.transform(image)

        return image,label

In [19]:
train_dataset = PetDataset(train_ds,train_transform)
val_dataset = PetDataset(val_ds,val_transform)
test_dataset = PetDataset(test_ds,test_transform)

print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

2944
736
3669


In [20]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)


In [21]:

images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Image dtype:", images.dtype)
print("Label dtype:", labels.dtype)
print("First 5 labels:", labels[:5])


Images shape: torch.Size([32, 3, 224, 224])
Labels shape: torch.Size([32])
Image dtype: torch.float32
Label dtype: torch.int64
First 5 labels: tensor([30,  4, 20, 12, 31])


In [22]:
import torch
import torch.nn as nn

In [23]:
class SimpleCNN(nn.Module):
    def __init__(self,num_classes = 37):
        super().__init__()

        self.features = nn.Sequential(
            #Block 1
            nn.Conv2d(3,16,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            #Block2
            nn.Conv2d(16,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            #Block 3
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.pool = nn.AdaptiveAvgPool2d((1,1))

        self.classifier = nn.Linear(64,num_classes)

    def forward(self, x):
            x = self.features(x)
            x = self.pool(x)
            x = torch.flatten(x, start_dim=1)
            x = self.classifier(x)

            return x

In [24]:
'''
import torch
import torch.nn as nn
import copy

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Using device:", device)

model = SimpleCNN(num_classes=37).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

num_epochs = 10
'''

'\nimport torch\nimport torch.nn as nn\nimport copy\n\ndevice = torch.device(\n    "cuda" if torch.cuda.is_available()\n    else "mps" if torch.backends.mps.is_available()\n    else "cpu"\n)\n\nprint("Using device:", device)\n\nmodel = SimpleCNN(num_classes=37).to(device)\n\ncriterion = nn.CrossEntropyLoss()\noptimizer = torch.optim.Adam(\n    model.parameters(),\n    lr=0.001,\n    weight_decay=1e-4\n)\n\nnum_epochs = 10\n'

In [25]:

import torch
import copy

def fit(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    device,
    epochs=10,
    head_only=False,
    freeze_bn=False
):
    model = model.to(device)

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    best_val_acc = -1.0
    best_state = None

    for epoch in range(epochs):

        # ============== TRAINING ==============
        if head_only:
            model.eval()  # Keep frozen backbone BN statistics fixed
            model.get_classifier().train()
        else:
            model.train()

        if freeze_bn:
            for module in model.modules():
                if isinstance(module, nn.modules.batchnorm._BatchNorm):
                    module.eval()


        train_loss_sum = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * labels.size(0)
            train_correct += (outputs.argmax(1) == labels).sum().item()
            train_total += labels.size(0)

        train_loss = train_loss_sum / train_total
        train_acc = train_correct / train_total

        # ============== VALIDATION ==============
        model.eval()

        val_loss_sum = 0.0
        val_correct = 0
        val_total = 0

        with torch.inference_mode():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss_sum += loss.item() * labels.size(0)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_total += labels.size(0)

        val_loss = val_loss_sum / val_total
        val_acc = val_correct / val_total

        # ============== HISTORY ==============
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        # ============== BEST MODEL ==============
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

        print(
            f"Epoch [{epoch+1}/{epochs}] | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc*100:.2f}% | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc*100:.2f}%"
        )

    # Restore best validation checkpoint
    model.load_state_dict(best_state)

    print(f"\nBest Validation Accuracy: {best_val_acc*100:.2f}%")

    return history


In [26]:

import matplotlib.pyplot as plt

def plot_history(history):
    epochs = range(1, len(history["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, history["train_loss"], label="Train")
    axes[0].plot(epochs, history["val_loss"], label="Validation")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(epochs, history["train_acc"], label="Train")
    axes[1].plot(epochs, history["val_acc"], label="Validation")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


In [ ]:

from pathlib import Path
import torch
import timm

# Find the checkpoint wherever Kaggle mounted it
checkpoint_files = list(
    Path("/kaggle/input").rglob("resnet18_head_only_best.pth")
)

print("Found checkpoints:", checkpoint_files)

if len(checkpoint_files) != 1:
    raise FileNotFoundError(
        "Checkpoint not found uniquely. "
        "Verify that the Kaggle dataset is attached."
    )

checkpoint_path = checkpoint_files[0]

# Recreate ResNet18
model = timm.create_model(
    "resnet18",
    pretrained=False,
    num_classes=37
)

# Restore previously trained weights
state_dict = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

model.load_state_dict(state_dict)
model = model.to(device)

print("Checkpoint loaded successfully!")
print("Classifier:", model.get_classifier())


Checkpoint loaded successfully!
Classifier: Linear(in_features=512, out_features=37, bias=True)


In [28]:
model.requires_grad_(False)

model.layer4.requires_grad_(True)

model.get_classifier().requires_grad_(True)


# Verify trainable parameters
total_params = sum(p.numel() for p in model.parameters())

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")

Total parameters: 11,195,493
Trainable parameters: 8,412,709
Frozen parameters: 2,782,784


In [29]:

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    [
        {
            "params": model.layer4.parameters(),
            "lr": 1e-5
        },
        {
            "params": model.get_classifier().parameters(),
            "lr": 1e-4
        }
    ],
    weight_decay=1e-4
)


In [ ]:

partial_fine_tuned_model = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    epochs=5,
    head_only=False,
    freeze_bn=True
)

plot_history(partial_fine_tuned_model)


KeyboardInterrupt: 

In [ ]:

from pathlib import Path

save_path = Path("/kaggle/working/resnet18_layer4_best.pth")

torch.save(model.state_dict(), save_path)

print("Saved checkpoint:", save_path)
